# 02 · Video Acquisition and Dataset Audit

**Project:** Isolated word-level American Sign Language recognition  
**Dataset:** WLASL-100  
**Input:** Frozen metadata and signer-independent splits from Notebook 01

## Objective

Use the frozen metadata and downloaded WLASL instance videos when available, validate local videos by decoding frames, extract YouTube-backed instances, and report availability by the frozen train/validation/test split.

**Important distinctions**
- A YouTube source video may contain several WLASL sign instances.
- A non-YouTube URL generally refers to an individual WLASL instance.
- A download log entry is not proof that a file is usable. OpenCV validation is the authority.
- This notebook never regenerates the signer-independent split.

**Safe defaults:** Downloading and extraction are disabled. The first run audits what is already on disk. Enable acquisition only after reviewing the inventories.

## 1. Imports and project paths

Set `PROJECT_DIR` or `INSTANCE_VIDEO_DIR` only when automatic detection does not find your folders. The default external dataset folder is `~/Downloads/archive/videos`.


In [1]:
from pathlib import Path
from urllib.parse import urlparse, parse_qs
from datetime import datetime, timezone
import json
import shutil
import pandas as pd
import cv2
from IPython.display import display

PROJECT_DIR = None  # Example: Path.home() / "sign-language-recognition"
INSTANCE_VIDEO_DIR = Path.home() / "Downloads" / "archive" / "videos"  # WLASL videos named by 5-digit instance ID

def find_project_root(start):
    candidates = [start, *start.parents, Path.home() / "sign-language-recognition"]
    for candidate in candidates:
        if ((candidate / "data" / "processed" / "WLASL100" / "WLASL100_filtered.json").exists()
                or (candidate / "WLASL" / "start_kit" / "WLASL_v0.3.json").exists()):
            return candidate.resolve()
    return None

project_dir = Path(PROJECT_DIR).expanduser().resolve() if PROJECT_DIR else find_project_root(Path.cwd())
if project_dir is None:
    raise FileNotFoundError(
        f"Cannot locate project metadata (data/processed/WLASL100/WLASL100_filtered.json).\n"
        f"Current working directory: {Path.cwd()}\n"
        "Set PROJECT_DIR to the project folder containing data/processed/WLASL100."
    )

processed_dir = project_dir / "data" / "processed" / "WLASL100"
source_dir = project_dir / "data" / "raw" / "youtube_sources"
non_youtube_dir = project_dir / "data" / "raw" / "non_youtube_instances"
clip_dir = processed_dir / "clips"
instance_video_dir = Path(INSTANCE_VIDEO_DIR).expanduser() if INSTANCE_VIDEO_DIR else None

for folder in (processed_dir, source_dir, non_youtube_dir, clip_dir):
    folder.mkdir(parents=True, exist_ok=True)

filtered_json = processed_dir / "WLASL100_filtered.json"
split_files = {
    "train": processed_dir / "train_metadata.csv",
    "val": processed_dir / "val_metadata.csv",
    "test": processed_dir / "test_metadata.csv",
}
if not filtered_json.exists():
    raise FileNotFoundError(f"Missing {filtered_json}. Run Notebook 01 first.")

print("Project root:", project_dir)
print("YouTube sources:", source_dir)
print("Non-YouTube instances:", non_youtube_dir)
print("Downloaded WLASL instance videos:", instance_video_dir, "(found)" if instance_video_dir and instance_video_dir.is_dir() else "(not found)")
print("Final instance clips:", clip_dir)

Project root: /Users/sahilchauhan/sign-language-recognition
YouTube sources: /Users/sahilchauhan/sign-language-recognition/data/raw/youtube_sources
Non-YouTube instances: /Users/sahilchauhan/sign-language-recognition/data/raw/non_youtube_instances
Extracted clips: /Users/sahilchauhan/sign-language-recognition/data/processed/WLASL100/clips


## 2. Load the frozen WLASL-100 metadata

Keep `video_id` as a zero-padded string. This prevents IDs such as `07085` from being silently changed to `7085`.

In [2]:
with filtered_json.open("r") as f:
    filtered_data = json.load(f)

records = []
for entry in filtered_data:
    for instance in entry["instances"]:
        record = {"gloss": entry["gloss"], **instance}
        record["video_id"] = str(record["video_id"]).zfill(5)
        records.append(record)

instances_df = pd.DataFrame(records)

required = {"gloss", "video_id", "signer_id", "fps", "frame_start", "frame_end", "url"}
missing = required - set(instances_df.columns)
if missing:
    raise ValueError(f"Filtered metadata is missing columns: {sorted(missing)}")

assert len(instances_df) == 2038, f"Expected 2,038 WLASL-100 instances; found {len(instances_df)}"
assert instances_df["video_id"].is_unique, "Expected unique WLASL instance IDs."

print(f"Classes: {instances_df['gloss'].nunique():,}")
print(f"Instances: {len(instances_df):,}")
print(f"Signers: {instances_df['signer_id'].nunique():,}")
display(instances_df[["gloss", "video_id", "signer_id", "url"]].head())

Classes: 100
Instances: 2,038
Signers: 97


,gloss,video_id,signer_id,url
0,book,69241,118,http://aslbricks.org/New/ASL-Videos/book.mp4
1,book,65225,90,https://aslsignbank.haskins.yale.edu/dictionar...
2,book,68011,110,https://www.youtube.com/watch?v=0UsjUE-TXns
3,book,68208,113,https://www.youtube.com/watch?v=1QOYOZ3g-aY
4,book,68012,109,https://www.youtube.com/watch?v=aGtIHKEdCds


## 3. Classify YouTube and non-YouTube instances

YouTube source IDs are deduplicated so each source is downloaded once, even when it supports multiple WLASL instances.

In [3]:
def extract_youtube_id(url):
    if not isinstance(url, str) or not url.strip():
        return None
    parsed = urlparse(url)
    host = parsed.netloc.lower().split(":")[0]
    if host == "youtu.be" or host.endswith(".youtu.be"):
        return parsed.path.strip("/").split("/")[0] or None
    if host == "youtube.com" or host.endswith(".youtube.com"):
        parts = parsed.path.strip("/").split("/")
        if parts and parts[0] in {"shorts", "embed", "live"} and len(parts) > 1:
            return parts[1]
        return parse_qs(parsed.query).get("v", [None])[0]
    return None

instances_df["youtube_id"] = instances_df["url"].apply(extract_youtube_id)
youtube_instances = instances_df[instances_df["youtube_id"].notna()].copy()
non_youtube_instances = instances_df[instances_df["youtube_id"].isna()].copy()

youtube_inventory = (
    youtube_instances.groupby("youtube_id")
    .agg(
        instance_count=("video_id", "count"),
        class_count=("gloss", "nunique"),
        glosses=("gloss", lambda s: ", ".join(sorted(set(s))))
    )
    .reset_index()
)

print(f"Total WLASL-100 instances: {len(instances_df):,}")
print(f"YouTube instances: {len(youtube_instances):,}")
print(f"Unique YouTube source videos: {youtube_inventory['youtube_id'].nunique():,}")
print(f"Non-YouTube instances: {len(non_youtube_instances):,}")
print(f"Unique non-YouTube URLs: {non_youtube_instances['url'].nunique():,}")
assert len(youtube_instances) + len(non_youtube_instances) == len(instances_df)
display(pd.DataFrame({
    "type": ["YouTube", "Non-YouTube"],
    "instances": [len(youtube_instances), len(non_youtube_instances)]
}))

Total WLASL-100 instances: 2,038
YouTube instances: 685
Unique YouTube source videos: 572
Non-YouTube instances: 1,353
Unique non-YouTube URLs: 1,341


,type,instances
0,YouTube,685
1,Non-YouTube,1353


## 4. Video validation utility

A file is valid only if OpenCV opens it, reports sensible metadata, and successfully decodes frames near the beginning, middle, and end. File size alone is never sufficient.

In [4]:
def audit_video(path):
    path = Path(path)
    result = {
        "path": str(path),
        "exists": path.is_file(),
        "valid": False,
        "width": None,
        "height": None,
        "fps": None,
        "frame_count": None,
        "duration_sec": None,
        "size_mb": round(path.stat().st_size / (1024 ** 2), 3) if path.is_file() else 0.0,
        "error": None,
    }
    if not result["exists"] or path.stat().st_size == 0:
        return result

    cap = cv2.VideoCapture(str(path))
    try:
        if not cap.isOpened():
            result["error"] = "opencv_open_failed"
            return result

        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = float(cap.get(cv2.CAP_PROP_FPS))
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        result.update({
            "width": width, "height": height, "fps": fps,
            "frame_count": frame_count,
            "duration_sec": frame_count / fps if fps > 0 else None,
        })

        if width <= 0 or height <= 0 or fps <= 0 or frame_count <= 0:
            result["error"] = "invalid_video_metadata"
            return result

        decoded = []
        for fraction in (0.0, 0.5, 0.95):
            frame_index = min(int(frame_count * fraction), frame_count - 1)
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_index)
            ok, frame = cap.read()
            decoded.append(bool(ok and frame is not None and frame.size))
        result["valid"] = all(decoded)
        if not result["valid"]:
            result["error"] = "frame_decode_failed"
        return result
    finally:
        cap.release()

def find_video_file(folder, video_id, extensions=(".mp4", ".mov", ".mkv", ".avi", ".webm")):
    folder = Path(folder)
    for ext in extensions:
        candidate = folder / f"{video_id}{ext}"
        if candidate.is_file() and candidate.stat().st_size > 0:
            return candidate
    # Handle unexpected extension/case without assuming every matching file is a video.
    matches = sorted(p for p in folder.glob(f"{video_id}.*") if p.is_file() and p.suffix.lower() in extensions)
    return matches[0] if matches else None

## 5. Audit existing YouTube source files

This audit checks actual files in `data/raw/youtube_sources`. The downloaded WLASL instance videos are audited separately and can supply final clips directly.


In [5]:
youtube_rows = []
for video_id in youtube_inventory["youtube_id"]:
    path = find_video_file(source_dir, video_id)
    if path is None:
        youtube_rows.append({
            "youtube_id": video_id, "path": None, "exists": False, "valid": False,
            "width": None, "height": None, "fps": None, "frame_count": None,
            "duration_sec": None, "size_mb": 0.0, "error": "file_not_found"
        })
    else:
        youtube_rows.append({"youtube_id": video_id, **audit_video(path)})

youtube_audit = pd.DataFrame(youtube_rows)
youtube_inventory = youtube_inventory.merge(
    youtube_audit[["youtube_id", "path", "exists", "valid", "size_mb", "error"]],
    on="youtube_id", how="left"
)

valid_source_ids = set(youtube_inventory.loc[youtube_inventory["valid"], "youtube_id"])
print(f"Unique YouTube sources: {len(youtube_inventory):,}")
print(f"Files present: {int(youtube_inventory['exists'].sum()):,}")
print(f"Validated sources: {int(youtube_inventory['valid'].sum()):,}")
print(f"Missing/invalid sources: {int((~youtube_inventory['valid']).sum()):,}")
print(
    "Metadata instances backed by validated sources:",
    int(youtube_inventory.loc[youtube_inventory["valid"], "instance_count"].sum())
)
display(youtube_inventory["valid"].value_counts().rename_axis("valid").to_frame("source_count"))

Unique YouTube sources: 572
Files present: 340
Validated sources: 340
Missing/invalid sources: 232
Metadata instances backed by validated sources: 453


,source_count
valid,
True,340
False,232


## 6. Optional YouTube acquisition

Downloading is disabled by default. Enable it only if you want to fetch missing YouTube source videos after reviewing the inventories.


In [15]:
RUN_YOUTUBE_DOWNLOADS = False
MAX_NEW_SOURCES = 20  # Set to None to process all eligible sources in one run.
RETRY_PRIVATE_OR_UNAVAILABLE = True

if RUN_YOUTUBE_DOWNLOADS:
    import yt_dlp

    log_path = source_dir / "youtube_download_log.json"
    download_log = json.loads(log_path.read_text()) if log_path.exists() else {}

    pending_rows = []
    for row in youtube_inventory.itertuples(index=False):
        if bool(row.valid):
            continue
        previous = download_log.get(row.youtube_id, {})
        previous_status = str(previous.get("status", "")).lower()
        terminal = ("private video" in previous_status) or ("unavailable" in previous_status)
        if terminal and not RETRY_PRIVATE_OR_UNAVAILABLE:
            continue
        pending_rows.append(row)

    pending_rows = sorted(pending_rows, key=lambda r: r.instance_count, reverse=True)
    if MAX_NEW_SOURCES is not None:
        pending_rows = pending_rows[:MAX_NEW_SOURCES]

    ydl_options = {
        "format": "bestvideo+bestaudio/best",
        "outtmpl": str(source_dir / "%(id)s.%(ext)s"),
        "merge_output_format": "mp4",
        "js_runtimes": {"node": {}},
        "remote_components": {"ejs:github"},
        "quiet": True,
        "no_warnings": False,
        "noplaylist": True,
        "retries": 2,
        "fragment_retries": 2,
        "overwrites": False,
    }

    print(f"Attempting {len(pending_rows)} source downloads.")
    for row in pending_rows:
        video_id = row.youtube_id
        status = "unknown"
        try:
            with yt_dlp.YoutubeDL(ydl_options) as ydl:
                ydl.download([f"https://www.youtube.com/watch?v={video_id}"])

            path = find_video_file(source_dir, video_id)
            check = audit_video(path) if path else {"valid": False, "error": "file_not_found"}
            status = "downloaded_and_validated" if check["valid"] else f"downloaded_but_invalid:{check.get('error')}"
        except Exception as exc:
            status = f"failed:{str(exc)[:500]}"

        download_log[video_id] = {
            "status": status,
            "instance_count": int(row.instance_count),
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        }
        log_path.write_text(json.dumps(download_log, indent=2))
        print(video_id, status)
else:
    print("YouTube downloading is OFF. Existing files were only audited.")

Attempting 20 source downloads.


ERROR: [youtube] --Xi-n0DgBU: This video is unavailable


--Xi-n0DgBU failed:ERROR: [youtube] --Xi-n0DgBU: This video is unavailable


ERROR: [youtube] -9AsKTaXwH8: Private video. If the owner of this video has granted you access, please sign in. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies


-9AsKTaXwH8 failed:ERROR: [youtube] -9AsKTaXwH8: Private video. If the owner of this video has granted you access, please sign in. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies


ERROR: [youtube] -IWU7gZ-gAQ: This video is unavailable


... (31 outputs left out)

ERROR: [youtube] 2x5hblbX1Ho: Private video. If the owner of this video has granted you access, please sign in. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies


2x5hblbX1Ho failed:ERROR: [youtube] 2x5hblbX1Ho: Private video. If the owner of this video has granted you access, please sign in. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies


ERROR: [youtube] 3-GrhsVs830: Private video. If the owner of this video has granted you access, please sign in. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies


3-GrhsVs830 failed:ERROR: [youtube] 3-GrhsVs830: Private video. If the owner of this video has granted you access, please sign in. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies


## 7. Build the non-YouTube instance inventory

Non-YouTube URLs are instance-level sources. The downloaded WLASL collection (5-digit filenames) can provide clips for any matching WLASL instance, regardless of its original URL.


In [7]:
non_youtube_inventory = non_youtube_instances[
    ["video_id", "gloss", "signer_id", "url", "fps", "frame_start", "frame_end"]
].copy()

non_youtube_json_data = []
non_youtube_id_set = set(non_youtube_inventory["video_id"])
for entry in filtered_data:
    chosen = [
        instance for instance in entry["instances"]
        if str(instance["video_id"]).zfill(5) in non_youtube_id_set
    ]
    if chosen:
        non_youtube_json_data.append({**entry, "instances": chosen})

non_youtube_json_path = processed_dir / "WLASL100_non_youtube.json"
with non_youtube_json_path.open("w") as f:
    json.dump(non_youtube_json_data, f, indent=2)

json_count = sum(len(entry["instances"]) for entry in non_youtube_json_data)
assert json_count == len(non_youtube_inventory)

print(f"Non-YouTube instances: {len(non_youtube_inventory):,}")
print(f"Unique non-YouTube URLs: {non_youtube_inventory['url'].nunique():,}")
print("Filtered downloader JSON:", non_youtube_json_path)

Non-YouTube instances: 1,353
Unique non-YouTube URLs: 1,341
Filtered downloader JSON: /Users/sahilchauhan/sign-language-recognition/data/processed/WLASL100/WLASL100_non_youtube.json


## 8. Optional official non-YouTube downloader

This network downloader is disabled by default. Local downloaded WLASL instance videos are used below when available.


In [ ]:
RUN_NON_YOUTUBE_DOWNLOADS = False

if RUN_NON_YOUTUBE_DOWNLOADS:
    import importlib.util

    downloader_path = project_dir / "WLASL" / "start_kit" / "video_downloader.py"
    if not downloader_path.exists():
        raise FileNotFoundError(f"Official downloader not found: {downloader_path}")

    spec = importlib.util.spec_from_file_location("wlasl_downloader", downloader_path)
    wlasl_downloader = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(wlasl_downloader)

    wlasl_downloader.download_nonyt_videos(
        str(non_youtube_json_path),
        saveto=str(non_youtube_dir)
    )
else:
    print("Non-YouTube downloading is OFF. Existing instance files will be audited.")

gloss: book, video: 69241.
69241 exists at /Users/sahilchauhan/sign-language-recognition/data/raw/non_youtube_instances/69241.mp4
gloss: book, video: 65225.
Requesting https://aslsignbank.haskins.yale.edu/dictionary/protected_media/glossvideo/ASL/BO/BOOK-418.mp4
Unsuccessful downloading - video 65225
gloss: book, video: 07085.
Requesting http://www.aslpro.com/main/b/book_english_grammar.swf
Unsuccessful downloading - video 07085
gloss: book, video: 07086.
Requesting http://www.aslpro.com/main/b/book_geography.swf
Unsuccessful downloading - video 07086
gloss: book, video: 07087.
Requesting http://www.aslpro.com/main/b/book_geometry.swf
Unsuccessful downloading - video 07087
gloss: book, video: 07069.
07069 exists at /Users/sahilchauhan/sign-language-recognition/data/raw/non_youtube_instances/07069.mp4
gloss: book, video: 07088.
Requesting http://www.aslpro.com/main/b/book_history.swf
Unsuccessful downloading - video 07088
gloss: book, video: 07089.
Requesting http://www.aslpro.com/main/

## 9. Audit existing non-YouTube files

Searches for each WLASL ID across common video extensions instead of assuming every file is `.mp4`. Flash `.swf` files are not treated as decoded video clips by OpenCV; they require the dataset's legacy conversion workflow and are reported separately.

In [9]:
non_youtube_rows = []
for row in non_youtube_inventory.itertuples(index=False):
    video_id = row.video_id
    path = find_video_file(instance_video_dir, video_id) if instance_video_dir and instance_video_dir.is_dir() else None
    if path is None:
        path = find_video_file(non_youtube_dir, video_id)
    if path is None:
        non_youtube_rows.append({
            "video_id": video_id, "gloss": row.gloss, "url": row.url,
            "path": None, "exists": False, "valid": False, "size_mb": 0.0,
            "error": "file_not_found", "extension": None
        })
    else:
        info = audit_video(path)
        non_youtube_rows.append({
            "video_id": video_id, "gloss": row.gloss, "url": row.url,
            **info, "extension": path.suffix.lower()
        })

non_youtube_audit = pd.DataFrame(non_youtube_rows)
print(f"Non-YouTube metadata instances: {len(non_youtube_audit):,}")
print(f"Files present: {int(non_youtube_audit['exists'].sum()):,}")
print(f"Validated videos: {int(non_youtube_audit['valid'].sum()):,}")
print(f"Missing/invalid: {int((~non_youtube_audit['valid']).sum()):,}")
if non_youtube_audit["exists"].any():
    display(non_youtube_audit.loc[non_youtube_audit["exists"], ["video_id", "gloss", "extension", "valid", "error"]].head(20))
display(non_youtube_audit["valid"].value_counts().rename_axis("valid").to_frame("instance_count"))

Non-YouTube metadata instances: 1,353
Files present: 4
Validated videos: 4
Missing/invalid: 1,349


,video_id,gloss,extension,valid,error
0,69241,book,.mp4,True,NaN
5,07069,book,.mp4,True,NaN
12,07068,book,.mp4,True,NaN
17,07070,book,.mp4,True,NaN


,instance_count
valid,
False,1349
True,4


## 10. Extract YouTube-backed WLASL instance clips

If the instance-level WLASL download already contains a matching ID, it is used directly by the consolidation step. Otherwise, use frozen metadata frame boundaries to extract the sign from its YouTube source.


In [10]:
def extract_instance_clip(row, overwrite=False):
    video_id = str(row["video_id"]).zfill(5)
    source_path = find_video_file(source_dir, row["youtube_id"])
    output_path = clip_dir / f"{video_id}.mp4"
    temp_path = clip_dir / f"{video_id}.tmp.mp4"

    if output_path.exists() and not overwrite and audit_video(output_path)["valid"]:
        return {"video_id": video_id, "gloss": row["gloss"], "status": "already_valid"}

    if source_path is None:
        return {"video_id": video_id, "gloss": row["gloss"], "status": "source_missing"}
    if not audit_video(source_path)["valid"]:
        return {"video_id": video_id, "gloss": row["gloss"], "status": "source_invalid"}

    cap = cv2.VideoCapture(str(source_path))
    try:
        source_fps = float(cap.get(cv2.CAP_PROP_FPS))
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        metadata_fps = float(row["fps"])
        if source_fps <= 0 or metadata_fps <= 0 or total_frames <= 0:
            return {"video_id": video_id, "gloss": row["gloss"], "status": "invalid_source_metadata"}

        start = max(0, round((int(row["frame_start"]) - 1) * source_fps / metadata_fps))
        if int(row["frame_end"]) == -1:
            end = total_frames - 1
        else:
            end = round(int(row["frame_end"]) * source_fps / metadata_fps) - 1
        end = min(end, total_frames - 1)

        if start >= total_frames or end < start:
            return {"video_id": video_id, "gloss": row["gloss"], "status": "invalid_frame_range"}

        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        writer = cv2.VideoWriter(
            str(temp_path), cv2.VideoWriter_fourcc(*"mp4v"),
            source_fps, (width, height)
        )
        if not writer.isOpened():
            writer.release()
            return {"video_id": video_id, "gloss": row["gloss"], "status": "writer_open_failed"}

        cap.set(cv2.CAP_PROP_POS_FRAMES, start)
        expected = end - start + 1
        written = 0
        try:
            for _ in range(expected):
                ok, frame = cap.read()
                if not ok:
                    break
                writer.write(frame)
                written += 1
        finally:
            writer.release()

        temp_check = audit_video(temp_path)
        if written == expected and temp_check["valid"]:
            temp_path.replace(output_path)
            return {
                "video_id": video_id, "gloss": row["gloss"], "status": "success",
                "frames_written": written, "expected_frames": expected
            }

        temp_path.unlink(missing_ok=True)
        return {
            "video_id": video_id, "gloss": row["gloss"], "status": "incomplete_or_invalid",
            "frames_written": written, "expected_frames": expected
        }
    finally:
        cap.release()

RUN_EXTRACTION = False
MAX_CLIPS = 20  # Set None to process all eligible YouTube instances.

if RUN_EXTRACTION:
    eligible = youtube_instances[
        youtube_instances["youtube_id"].isin(valid_source_ids)
    ].copy()
    eligible = eligible.sort_values(["gloss", "video_id"])
    if MAX_CLIPS is not None:
        eligible = eligible.head(MAX_CLIPS)

    extraction_results = [
        extract_instance_clip(row)
        for _, row in eligible.iterrows()
    ]
    extraction_df = pd.DataFrame(extraction_results)
    display(extraction_df["status"].value_counts().rename_axis("status").to_frame("count"))
else:
    print("Clip extraction is OFF.")

Clip extraction is OFF.


## 11. Consolidate downloaded WLASL instance videos

Validated 5-digit instance videos from `INSTANCE_VIDEO_DIR` are copied into `clips/` for IDs in the frozen metadata. Existing valid clips are preserved. Missing IDs can still be supplied by validated non-YouTube downloads.


In [11]:
# Copies only metadata-matched, decodable WLASL clips into the project.
COPY_DOWNLOADED_INSTANCE_VIDEOS = True

if COPY_DOWNLOADED_INSTANCE_VIDEOS and instance_video_dir and instance_video_dir.is_dir():
    copied = 0
    for row in instances_df.itertuples(index=False):
        video_id = str(row.video_id).zfill(5)
        source_path = find_video_file(instance_video_dir, video_id, extensions=(".mp4", ".mov", ".mkv", ".avi", ".webm"))
        if source_path is None or not audit_video(source_path)["valid"]:
            continue
        destination = clip_dir / f"{video_id}.mp4"
        if destination.exists() and audit_video(destination)["valid"]:
            continue
        temp_path = clip_dir / f"{video_id}.import.tmp.mp4"
        shutil.copy2(source_path, temp_path)
        if audit_video(temp_path)["valid"]:
            temp_path.replace(destination)
            copied += 1
        else:
            temp_path.unlink(missing_ok=True)
    print(f"Validated downloaded instance clips copied: {copied}")
else:
    print("Downloaded instance video import is OFF or folder is unavailable.")

# Also consolidate validated non-YouTube MP4s not already supplied by the downloaded collection.
COPY_VALID_NON_YOUTUBE_MP4 = True
if COPY_VALID_NON_YOUTUBE_MP4:
    copied = 0
    for row in non_youtube_audit.itertuples(index=False):
        if not row.valid or row.extension != ".mp4":
            continue
        source_path = Path(row.path)
        destination = clip_dir / f"{row.video_id}.mp4"
        if destination.exists() and audit_video(destination)["valid"]:
            continue
        shutil.copy2(source_path, destination)
        if audit_video(destination)["valid"]:
            copied += 1
        else:
            destination.unlink(missing_ok=True)
    print(f"Validated non-YouTube MP4 clips copied: {copied}")


Copying non-YouTube MP4 instances is OFF.


## 12. Audit final instance clips

This audit validates the common `clips/` directory independently of download/extraction logs.

In [12]:
clip_rows = []
for row in instances_df.itertuples(index=False):
    video_id = row.video_id
    path = find_video_file(clip_dir, video_id, extensions=(".mp4",))
    if path is None:
        clip_rows.append({
            "video_id": video_id, "gloss": row.gloss, "signer_id": row.signer_id,
            "path": None, "exists": False, "valid": False, "size_mb": 0.0,
            "error": "clip_not_found"
        })
    else:
        clip_rows.append({
            "video_id": video_id, "gloss": row.gloss, "signer_id": row.signer_id,
            **audit_video(path)
        })

clip_audit = pd.DataFrame(clip_rows)
valid_clips = clip_audit[clip_audit["valid"]].copy()
print(f"Metadata instances: {len(instances_df):,}")
print(f"Clip files present: {int(clip_audit['exists'].sum()):,}")
print(f"Validated instance clips: {len(valid_clips):,}")
print(f"Classes represented: {valid_clips['gloss'].nunique():,}")
display(clip_audit["valid"].value_counts().rename_axis("valid").to_frame("instance_count"))

Metadata instances: 2,038
Clip files present: 0
Validated instance clips: 0
Classes represented: 0


,instance_count
valid,
False,2038


## 13. Availability by frozen signer-independent split

Notebook 01 created the split assignments. This section joins the validated clip IDs to those exact CSVs. It does not create or modify partitions.

A class with zero validated examples in any partition is a blocker for the planned 100-class evaluation and must be addressed before model training.

In [13]:
split_frames = []
for split_name, path in split_files.items():
    if not path.exists():
        raise FileNotFoundError(f"Frozen {split_name} split not found: {path}")
    part = pd.read_csv(path, dtype={"video_id": str})
    part["video_id"] = part["video_id"].str.zfill(5)
    part["split"] = split_name
    split_frames.append(part[["video_id", "gloss", "signer_id", "split"]])

split_df = pd.concat(split_frames, ignore_index=True)
if split_df["video_id"].duplicated().any():
    raise ValueError("A video_id appears in more than one frozen split.")

availability = split_df.merge(
    clip_audit[["video_id", "valid"]].rename(columns={"valid": "clip_valid"}),
    on="video_id", how="left"
)
availability["clip_valid"] = availability["clip_valid"].fillna(False).astype(bool)

split_summary = (
    availability.groupby("split")
    .agg(metadata_instances=("video_id", "count"),
         validated_clips=("clip_valid", "sum"),
         classes_in_metadata=("gloss", "nunique"),
         classes_with_valid_clips=("gloss", lambda s: s[availability.loc[s.index, "clip_valid"]].nunique()))
    .reset_index()
)
split_summary["coverage_pct"] = (
    100 * split_summary["validated_clips"] / split_summary["metadata_instances"]
).round(2)

class_split_coverage = (
    availability.groupby(["split", "gloss"])["clip_valid"]
    .sum().unstack("split", fill_value=0)
)
print("Split-wise availability:")
display(split_summary)
print("Classes with zero valid clips in each split:")
for split_name in ("train", "val", "test"):
    zero_classes = class_split_coverage.index[class_split_coverage[split_name] == 0].tolist()
    print(f"{split_name}: {len(zero_classes)} classes")
    if zero_classes:
        print(zero_classes)

Split-wise availability:


,split,metadata_instances,validated_clips,classes_in_metadata,classes_with_valid_clips,coverage_pct
0,test,611,0,100,0,0.0
1,train,1003,0,100,0,0.0
2,val,424,0,100,0,0.0


Classes with zero valid clips in each split:
train: 100 classes
['accident', 'africa', 'all', 'apple', 'basketball', 'bed', 'before', 'bird', 'birthday', 'black', 'blue', 'book', 'bowling', 'brown', 'but', 'can', 'candy', 'chair', 'change', 'cheat', 'city', 'clothes', 'color', 'computer', 'cook', 'cool', 'corn', 'cousin', 'cow', 'dance', 'dark', 'deaf', 'decide', 'doctor', 'dog', 'drink', 'eat', 'enjoy', 'family', 'fine', 'finish', 'fish', 'forget', 'full', 'give', 'go', 'graduate', 'hat', 'hearing', 'help', 'hot', 'how', 'jacket', 'kiss', 'language', 'last', 'later', 'letter', 'like', 'man', 'many', 'medicine', 'meet', 'mother', 'need', 'no', 'now', 'orange', 'paint', 'paper', 'pink', 'pizza', 'play', 'pull', 'purple', 'right', 'same', 'school', 'secretary', 'shirt', 'short', 'son', 'study', 'table', 'tall', 'tell', 'thanksgiving', 'thin', 'thursday', 'time', 'walk', 'want', 'what', 'white', 'who', 'woman', 'work', 'wrong', 'year', 'yes']
val: 100 classes
['accident', 'africa', 'all',

## 14. Save acquisition reports and summary

All reports are saved under `data/processed/WLASL100/`. The JSON summary separates source availability from instance-clip availability.

In [14]:
youtube_inventory_path = processed_dir / "youtube_source_inventory.csv"
youtube_audit_path = processed_dir / "youtube_source_audit.csv"
non_youtube_inventory_path = processed_dir / "non_youtube_inventory.csv"
non_youtube_audit_path = processed_dir / "non_youtube_instance_audit.csv"
clip_audit_path = processed_dir / "instance_clip_audit.csv"
availability_path = processed_dir / "split_clip_availability.csv"
split_summary_path = processed_dir / "split_availability_summary.csv"
summary_path = processed_dir / "acquisition_summary.json"

youtube_inventory.to_csv(youtube_inventory_path, index=False)
youtube_audit.to_csv(youtube_audit_path, index=False)
non_youtube_inventory.to_csv(non_youtube_inventory_path, index=False)
non_youtube_audit.to_csv(non_youtube_audit_path, index=False)
clip_audit.to_csv(clip_audit_path, index=False)
availability.to_csv(availability_path, index=False)
split_summary.to_csv(split_summary_path, index=False)

summary = {
    "subset": "WLASL-100",
    "metadata_classes": int(instances_df["gloss"].nunique()),
    "metadata_instances": int(len(instances_df)),
    "youtube_instances": int(len(youtube_instances)),
    "unique_youtube_sources": int(youtube_inventory["youtube_id"].nunique()),
    "validated_youtube_sources": int(youtube_inventory["valid"].sum()),
    "non_youtube_instances": int(len(non_youtube_instances)),
    "validated_non_youtube_instances": int(non_youtube_audit["valid"].sum()),
    "validated_final_instance_clips": int(clip_audit["valid"].sum()),
    "classes_with_validated_clips": int(valid_clips["gloss"].nunique()),
    "split_summary": split_summary.to_dict(orient="records"),
    "generated_utc": datetime.now(timezone.utc).isoformat(),
}
summary_path.write_text(json.dumps(summary, indent=2))

print("Reports saved:")
for path in (
    youtube_inventory_path, youtube_audit_path, non_youtube_inventory_path,
    non_youtube_audit_path, clip_audit_path, availability_path,
    split_summary_path, summary_path
):
    print(" -", path)
display(pd.Series({k: v for k, v in summary.items() if k != "split_summary"}, name="value").to_frame())

Reports saved:
 - /Users/sahilchauhan/sign-language-recognition/data/processed/WLASL100/youtube_source_inventory.csv
 - /Users/sahilchauhan/sign-language-recognition/data/processed/WLASL100/youtube_source_audit.csv
 - /Users/sahilchauhan/sign-language-recognition/data/processed/WLASL100/non_youtube_inventory.csv
 - /Users/sahilchauhan/sign-language-recognition/data/processed/WLASL100/non_youtube_instance_audit.csv
 - /Users/sahilchauhan/sign-language-recognition/data/processed/WLASL100/instance_clip_audit.csv
 - /Users/sahilchauhan/sign-language-recognition/data/processed/WLASL100/split_clip_availability.csv
 - /Users/sahilchauhan/sign-language-recognition/data/processed/WLASL100/split_availability_summary.csv
 - /Users/sahilchauhan/sign-language-recognition/data/processed/WLASL100/acquisition_summary.json


,value
subset,WLASL-100
metadata_classes,100
metadata_instances,2038
youtube_instances,685
unique_youtube_sources,572
validated_youtube_sources,340
non_youtube_instances,1353
validated_non_youtube_instances,4
validated_final_instance_clips,0
classes_with_validated_clips,0


## 15. Methodological notes and handoff

1. **Source availability:** WLASL metadata may point to private, removed, restricted, or legacy-hosted videos. Report observed availability rather than assuming all 2,038 instances can be recovered.
2. **Source versus instance:** One YouTube source may back multiple instance records. Count sources and sign instances separately.
3. **Non-YouTube formats:** Legacy Flash `.swf` assets are not standard video files for OpenCV. Keep them separate until a documented conversion is performed.
4. **Temporal alignment:** YouTube instance boundaries are converted from metadata FPS to the actual downloaded source FPS.
5. **Frozen evaluation split:** Never regenerate the train/validation/test split in this notebook. Availability is joined to Notebook 01's saved assignments.
6. **Next notebook:** Preprocess only validated clips, and retain `video_id`, `gloss`, `signer_id`, and `split` throughout the pipeline.